# 第11回：OpenMMで分子の動きを観察する

演習の目安は20分。採点・提出はありません。上から順にセルを実行します。CPUランタイムで動きます。

Learning ResourcesのOpenMM教材を再利用します。ACE–ALA–NMEと水を含む1291原子の準備済み座標を同梱しました。入力の水素・水の追加作業は不要です。**平衡化済みの構造ではありません**。短い5 psの計算でコードと出力の対応を確認し、平衡への収束やタンパク質の折り畳みを判定しません。

In [ ]:
%pip -q install openmm==8.6.1 numpy matplotlib

In [ ]:
import base64
from pathlib import Path
PDB_DATA = "..."
Path("prepared.pdb").write_bytes(base64.b64decode(PDB_DATA))
print("準備済み入力を展開しました")

## 1. 条件を読む

力場はamber14-all、水はTIP3P。周期境界、PME、短距離カットオフ1 nm、水素を含む結合の拘束を使います。温度300 K、摩擦係数1/ps、時間刻み2 fs。LangevinMiddleIntegratorが摩擦と熱揺らぎを含む運動を数値積分します。

In [ ]:
import openmm as mm
from openmm import app, unit
import numpy as np, matplotlib.pyplot as plt, json, time, io, zipfile
OUT=Path("results"); OUT.mkdir(exist_ok=True)
pdb=app.PDBFile('prepared.pdb')
ff=app.ForceField('amber14-all.xml','amber14/tip3p.xml')
system=ff.createSystem(pdb.topology, nonbondedMethod=app.PME, nonbondedCutoff=1*unit.nanometer, constraints=app.HBonds)
TEMPERATURE=300
STEPS=2500
DT_PS=0.002
integrator=mm.LangevinMiddleIntegrator(TEMPERATURE*unit.kelvin, 1/unit.picosecond, DT_PS*unit.picosecond)
integrator.setRandomNumberSeed(101)
sim=app.Simulation(pdb.topology,system,integrator,mm.Platform.getPlatformByName('CPU'),{'Threads':'2'})
sim.context.setPositions(pdb.positions)
print(f'{system.getNumParticles()} atoms, {STEPS*DT_PS:g} ps, {STEPS} updates')

## 2. 短い計算を実行する

最小化で局所的な衝突を緩め、初速度を与えてから動かします。最小化は熱平衡化とは異なります。100ステップごとに位置・温度・エネルギーを保存します。

In [ ]:
sim.minimizeEnergy(maxIterations=100)
sim.context.setVelocitiesToTemperature(TEMPERATURE*unit.kelvin,102)
sim.reporters.append(app.StateDataReporter(str(OUT/'log.csv'),100,step=True,time=True,potentialEnergy=True,temperature=True))
sim.reporters.append(app.DCDReporter(str(OUT/'trajectory.dcd'),100))
positions=[]; start=time.perf_counter()
for i in range(STEPS//100):
    sim.step(100)
    positions.append(sim.context.getState(positions=True).getPositions(asNumpy=True).value_in_unit(unit.nanometer))
positions=np.array(positions)
np.save(OUT/'positions_nm.npy',positions)
state=sim.context.getState(positions=True,energy=True)
with (OUT/'final.pdb').open('w') as f: app.PDBFile.writeFile(pdb.topology,state.getPositions(),f)
(OUT/'system.xml').write_text(mm.XmlSerializer.serialize(system))
(OUT/'integrator.xml').write_text(mm.XmlSerializer.serialize(integrator))
report={'version':mm.__version__,'platform':'CPU','atoms':system.getNumParticles(),'steps':STEPS,'dt_ps':DT_PS,'duration_ps':STEPS*DT_PS,'seconds':time.perf_counter()-start,'frames':len(positions),'seed':[101,102]}
(OUT/'metadata.json').write_text(json.dumps(report,indent=2));print(report)

## 3. 温度とエネルギーを読む

温度は300 K付近で揺らぎ、瞬間ごとに同じ値にはなりません。熱浴とのエネルギー交換があるため、力学的エネルギーの保存をNVEと同じ条件で要求しません。この短さでは平均値の収束も判定できません。

In [ ]:
data=np.loadtxt(OUT/'log.csv',delimiter=',',skiprows=1)
assert np.isfinite(data).all()
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(data[:,1],data[:,3]);axes[0].axhline(TEMPERATURE,color='gray',ls='--');axes[0].set(xlabel='time / ps',ylabel='temperature / K')
axes[1].plot(data[:,1],data[:,2]);axes[1].set(xlabel='time / ps',ylabel='potential energy / kJ mol$^{-1}$')
fig.tight_layout();fig.savefig(OUT/'temperature_energy.png',dpi=160);plt.show()
print('最後の半分の平均温度（収束値ではない）:',data[len(data)//2:,3].mean())

## 4. 同じペプチドの三つの時点を見る

水を表示から省き、ペプチドの重原子だけを描きます。原子は連続的に移動していますが、ここでは保存間隔0.2 psの静止スナップショットを比較します。結合は入力のトポロジーから描きます。

In [ ]:
atoms=list(pdb.topology.atoms())
heavy=[a.index for a in atoms if a.residue.name!='HOH' and a.element.symbol!='H']
bonds=[(a.index,b.index) for a,b in pdb.topology.bonds() if a.index in heavy and b.index in heavy]
fig=plt.figure(figsize=(13,4))
lo=positions[:,heavy,:].min((0,1));hi=positions[:,heavy,:].max((0,1));center=(lo+hi)/2;rad=max(hi-lo)/2+.05
for panel,k in enumerate([0,len(positions)//2,len(positions)-1],1):
    ax=fig.add_subplot(1,3,panel,projection='3d');p=positions[k]
    for a,b in bonds: ax.plot(*p[[a,b]].T,color='gray')
    ax.scatter(*p[heavy].T,c=[{'C':'#444444','N':'#235cb7','O':'#cc4422'}.get(atoms[i].element.symbol,'green') for i in heavy],s=45)
    ax.set(xlim=(center[0]-rad,center[0]+rad),ylim=(center[1]-rad,center[1]+rad),zlim=(center[2]-rad,center[2]+rad),title=f'{(k+1)*100*DT_PS:.1f} ps');ax.set_axis_off()
fig.tight_layout();fig.savefig(OUT/'snapshots.png',dpi=160);plt.show()

## 5. 条件を一つ変える・保存する

温度を変える場合は条件セルのTEMPERATUREを変更して、条件セルから下を再実行します。新しいSimulationを作るので計算を最初から比較できます。時間刻みを大きくする操作はここでは行いません。

results内の入力条件・軌跡・ログ・図をZIPへ保存します。Colabでは左のファイル欄からZIPをダウンロードできます。

In [ ]:
with zipfile.ZipFile('bioinfo11_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write('prepared.pdb')
    for p in OUT.iterdir():
        if p.is_file(): z.write(p)
print('bioinfo11_results.zip を保存しました')

出典：Learning Resources / OpenMM, examples/01_first_run.py、inputs/water.pdb。入力の元資料はOpenMMToolsのAlanineDipeptideExplicit。OpenMM公式資料：https://docs.openmm.org/latest/userguide/ 。入力の由来とライセンスは同梱のinputsを参照。